# Projections de population 2021-2070 (INSEE, Omphale 2022)

Région Auvergne-Rhône-Alpes : 12 départements, 11 scénarios.

Populations en **milliers d'habitants**.

In [1]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../reference/projections_omphale_aura"
FICHIERS = {
    os.path.basename(f).replace("projections_scenario_", "").replace(".xlsx", ""): f
    for f in sorted(glob.glob(f"{DATA}/*.xlsx"))
}
CENTRAL = "scenario_central"
print(len(FICHIERS), "scénarios :", list(FICHIERS))


def lire(scenario, feuille):
    """Lit une feuille (l'en-tête du tableau est à la ligne 6)."""
    df = pd.read_excel(FICHIERS[scenario], sheet_name=feuille, header=5, dtype={0: str})
    return df


def population(scenario, niveau="DEP"):
    """Population par année, format long : territoire, annee, pop."""
    df = lire(scenario, f"Population_{niveau}")
    libelle = f"LIB{niveau}"
    long = df.drop(columns=[niveau]).melt(id_vars=libelle, var_name="annee", value_name="pop")
    long["annee"] = long["annee"].str.replace("POP_", "").astype(int)
    return long.rename(columns={libelle: "territoire"})

11 scénarios : ['esperance_de_vie_basse', 'esperance_de_vie_haute', 'fecondite_basse', 'fecondite_haute', 'migrations_basses', 'migrations_hautes', 'population_agee', 'population_basse', 'population_haute', 'population_jeune', 'scenario_central']


## 1. Aperçu des données (scénario central)

In [2]:
lire(CENTRAL, "Population_DEP").iloc[:, [0, 1, 2, 5, 17, 32, 51]]

,DEP,LIBDEP,POP_2018,POP_2021,POP_2033,POP_2048,POP_2068
0,01,Ain,648,664,712,746,754
1,03,Allier,337,334,323,310,296
2,07,Ardèche,327,328,335,339,334
3,15,Cantal,145,144,141,136,130
4,26,Drôme,515,524,549,567,568
5,38,Isère,1264,1279,1332,1371,1373
6,42,Loire,763,767,780,785,781
7,43,Haute-Loire,228,228,228,227,222
8,63,Puy-de-Dôme,659,668,699,716,719
9,69,Rhône,1860,1890,2005,2089,2115


## 2. Population par département, scénario central

In [ ]:
pop = population(CENTRAL).pivot(index="annee", columns="territoire", values="pop")
ax = pop.plot(figsize=(11, 5), colormap="tab20")
ax.set_ylabel("population (milliers)")
ax.legend(ncol=3, fontsize=8)
plt.show()

## 3. Croissance 2021 → 2050 par département (scénario central)

In [ ]:
croissance = (pop.loc[2050] / pop.loc[2021] - 1) * 100
croissance = croissance.sort_values()
croissance.plot.barh(figsize=(8, 5), color=["tab:red" if v < 0 else "tab:green" for v in croissance])
plt.xlabel("évolution 2021 → 2050 (%)")
plt.show()

## 4. Comparaison des 11 scénarios (région)

In [ ]:
region = pd.concat(
    {nom: population(nom, "REG").set_index("annee")["pop"] for nom in FICHIERS}, axis=1
)
ax = region.loc[2021:].plot(figsize=(11, 5), colormap="tab20")
ax.set_ylabel("population de la région (milliers)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

Population de la région en 2021, 2050 et 2070 par scénario, et écart de 2050 avec le scénario central.

In [ ]:
synthese = pd.DataFrame({
    "pop_2021": region.loc[2021],
    "pop_2050": region.loc[2050],
    "pop_2070": region.loc[2070],
})
synthese["ecart_central_2050_%"] = (synthese["pop_2050"] / synthese.loc[CENTRAL, "pop_2050"] - 1) * 100
synthese.round(1).sort_values("pop_2050")

## 5. Structure par âge : 2018 vs 2070 (région, scénario central)

In [ ]:
age = lire(CENTRAL, "Structure_Age_REG").iloc[0]
tranches = [c.rsplit("_", 1)[0] for c in age.index if c.endswith("_2018") and c.startswith("[")]
structure = pd.DataFrame({
    "2018": [age[f"{t}_2018"] for t in tranches],
    "2070": [age[f"{t}_2070"] for t in tranches],
}, index=tranches)
structure.plot.bar(figsize=(11, 4))
plt.ylabel("% de la population")
plt.title(f"Âge moyen : {age['AGE_MOY_2018']} ans en 2018, {age['AGE_MOY_2070']} ans en 2070")
plt.show()

## 6. Part des 65 ans et plus par département

In [ ]:
sa = lire(CENTRAL, "Structure_Age_DEP").set_index("LIBDEP")
t65 = [t for t in tranches if int(t[1:].split(",")[0]) >= 65]
part65 = pd.DataFrame({
    "2018": sa[[f"{t}_2018" for t in t65]].sum(axis=1),
    "2070": sa[[f"{t}_2070" for t in t65]].sum(axis=1),
}).sort_values("2070")
part65.plot.barh(figsize=(8, 6))
plt.xlabel("% de la population ayant 65 ans ou plus")
plt.show()

## 7. Moteurs de la croissance 2018-2070

TCAM = taux de croissance annuel moyen. Il se décompose en solde migratoire (arrivées - départs) et solde naturel (naissances - décès).

In [ ]:
tc = lire(CENTRAL, "TCAM_contrib_DEP").set_index("LIBREG").sort_values("TCAM_2018_2070")
ax = tc[["Contrib_SM", "Contrib_SN"]].plot.barh(stacked=True, figsize=(8, 5))
ax.scatter(tc["TCAM_2018_2070"], range(len(tc)), color="black", zorder=3, label="TCAM total")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("points de % par an")
ax.legend(["total", "solde migratoire", "solde naturel"])
plt.show()